# Arriendo de viviendas en Madrid



Importación de las librerías

In [1]:
# Para la manipulación de datos
import pandas as pd
import numpy as np

# Servicio y driver de Chrome de Selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Las opciones que vamos a tener para buscar elementos
from selenium.webdriver.common.by import By

# Para cuando queramos mandar pulsaciones de teclado
from selenium.webdriver.common.keys import Keys

# Hacemos que espere
import time

# Importaciones para esperas explícitas (mejor práctica que time.sleep)
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Importamos undetected-chromedriver para evitar el captcha
import undetected_chromedriver as uc

# Importamos excepciones comunes de Selenium para manejo de errores
from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementClickInterceptedException

In [2]:
service = Service(ChromeDriverManager().install())

In [3]:
from selenium import webdriver

driver = webdriver.Chrome()
print(driver.capabilities["browserVersion"])
driver.quit()

146.0.7680.178


Creamos el driver para controlar el navegador

In [11]:
# Aquí empiezo ejecutando el robot

import undetected_chromedriver as uc

driver = uc.Chrome(
    version_main=146,  #Para que funcione con chrome 146
    browser_executable_path=r"C:\Program Files\Google\Chrome\Application\chrome.exe",
    service=service,
    use_subprocess=False,
    headless=False,
)

Accedemos a la página principal

In [5]:
url = "https://www.idealista.com/areas/alquiler-viviendas/?shape=%28%28my%7BuFdctUaW%3Fye%40gYwZaVkXoT%7Ef%40aw%40pQtSx%5CnMzXhIzc%40_SmApiAu_%40xY%29%29"
driver.get(url)

Hay un botón de Consentir datos personales

In [6]:
wait = WebDriverWait(driver, 10)

accept = wait.until(EC.element_to_be_clickable((
    By.XPATH, "//p[contains(@class, 'fc-button-label') and contains(text(), 'Consentir')]"
)))
accept.click()

TimeoutException: Message: 
Stacktrace:
	undetected_chromedriver!GetHandleVerifier [0xcbcdf3+10b03]
	undetected_chromedriver!GetHandleVerifier [0xcbcf24+10c34]
	undetected_chromedriver!(No symbol) [0xaa2120]
	undetected_chromedriver!(No symbol) [0xaeabca]
	undetected_chromedriver!(No symbol) [0xaeae6b]
	undetected_chromedriver!(No symbol) [0xb2d0b2]
	undetected_chromedriver!(No symbol) [0xb0db54]
	undetected_chromedriver!(No symbol) [0xb2a9a5]
	undetected_chromedriver!(No symbol) [0xb0d8a6]
	undetected_chromedriver!(No symbol) [0xae0229]
	undetected_chromedriver!(No symbol) [0xae0fe4]
	undetected_chromedriver!GetHandleVerifier [0xf248b9+2785c9]
	undetected_chromedriver!GetHandleVerifier [0xf1feb5+273bc5]
	undetected_chromedriver!GetHandleVerifier [0xf3e06b+291d7b]
	undetected_chromedriver!GetHandleVerifier [0xcd5cc8+299d8]
	undetected_chromedriver!GetHandleVerifier [0xcdd9fd+3170d]
	undetected_chromedriver!GetHandleVerifier [0xcc58c8+195d8]
	undetected_chromedriver!GetHandleVerifier [0

Hago una funcion para guardar los datos más importantes de cada propiedad: título, 

In [12]:

URL_BASE = "https://www.idealista.com/areas/alquiler-viviendas/pagina-{}?shape=%28%28my%7BuFdctUaW%3Fye%40gYwZaVkXoT%7Ef%40aw%40pQtSx%5CnMzXhIzc%40_SmApiAu_%40xY%29%29"

TOTAL_PAGINAS =24  # ajusta al total real

todos_los_datos = []

for pagina in range(1, TOTAL_PAGINAS + 1):
    driver.get(URL_BASE.format(pagina))
    
    wait = WebDriverWait(driver, 15)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "article.item")))
    time.sleep(3)  # pausa extra para evitar bloqueos

    publicaciones = driver.find_elements(By.CSS_SELECTOR, "article.item")

    for pub in publicaciones:
        try:
            titulo_el = pub.find_element(By.CSS_SELECTOR, "a.item-link")
            titulo = titulo_el.get_attribute("title").strip()
        except:
            titulo = "N/A"
        try:
            precio = pub.find_element(By.CSS_SELECTOR, "span.item-price").text.strip()
        except:
            precio = "N/A"

        # habitaciones, m2 y altura están todos en span.item-detail
        habitaciones = "N/A"
        m2 = "N/A"
        altura = "N/A"
        try:
            detalles = pub.find_elements(By.CSS_SELECTOR, "div.item-detail-char span.item-detail")
            for detalle in detalles:
                texto = detalle.text.strip()
                if "hab." in texto:
                    habitaciones = texto.replace("hab.", "").strip()
                elif "m²" in texto:
                    m2 = texto.replace("m²", "").strip()
                elif "Planta" in texto or "Bajo" in texto or "Entresuelo" in texto:
                    altura = texto
        except:
            pass

        todos_los_datos.append({
            "titulo": titulo,
            "precio": precio,
            "habitaciones": habitaciones,
            "m2": m2,
            "altura": altura
        })

    print(f"Página {pagina}/{TOTAL_PAGINAS} — total registros: {len(todos_los_datos)}")
    time.sleep(3)  # pausa entre páginas

print(f"✅ Total propiedades extraídas: {len(todos_los_datos)}")

Página 1/24 — total registros: 30
Página 2/24 — total registros: 60
Página 3/24 — total registros: 90
Página 4/24 — total registros: 120
Página 5/24 — total registros: 150
Página 6/24 — total registros: 180
Página 7/24 — total registros: 210
Página 8/24 — total registros: 240
Página 9/24 — total registros: 270
Página 10/24 — total registros: 300
Página 11/24 — total registros: 330
Página 12/24 — total registros: 360
Página 13/24 — total registros: 390
Página 14/24 — total registros: 420
Página 15/24 — total registros: 450
Página 16/24 — total registros: 480
Página 17/24 — total registros: 510
Página 18/24 — total registros: 540
Página 19/24 — total registros: 570
Página 20/24 — total registros: 600
Página 21/24 — total registros: 630
Página 22/24 — total registros: 660
Página 23/24 — total registros: 690
Página 24/24 — total registros: 695
✅ Total propiedades extraídas: 695


In [13]:
df_Pisos = pd.DataFrame(todos_los_datos)
df_Pisos.head()

,titulo,precio,habitaciones,m2,altura
0,"Piso en Calle de Jerónima Llorente, 74, Bellas...",1.900€/mes,3,75,Planta 1ª exterior con ascensor
1,"Dúplex en Calle del Doctor Santero, Bellas Vis...",5.300€/mes,4,192,N/A
2,"Piso en Calle de las Margaritas, 15, Berruguet...",800€/mes,1,27,Bajo exterior sin ascensor
3,"Ático en Calle de Almansa, 7, Bellas Vistas, M...",1.578€/mes,1,42,Planta 3ª exterior sin ascensor
4,"Dúplex en Calle Aranjuez, 5, Bellas Vistas, Ma...",1.150€/mes,1,55,Bajo exterior con ascensor


In [14]:
import re
def extraer_zona(titulo):
    partes = titulo.split(',')
    # filtrar partes que sean solo números o espacios
    partes_limpias = [p.strip() for p in partes if not re.match(r'^\s*\d+\s*$', p)]
    # descartar la primera parte (tipo + calle) y tomar el resto
    return ', '.join(partes_limpias[1:]).strip()

df_Pisos['zona'] = df_Pisos['titulo'].apply(extraer_zona)
df_Pisos.head(10)

,titulo,precio,habitaciones,m2,altura,zona
0,"Piso en Calle de Jerónima Llorente, 74, Bellas...",1.900€/mes,3,75,Planta 1ª exterior con ascensor,"Bellas Vistas, Madrid"
1,"Dúplex en Calle del Doctor Santero, Bellas Vis...",5.300€/mes,4,192,N/A,"Bellas Vistas, Madrid"
2,"Piso en Calle de las Margaritas, 15, Berruguet...",800€/mes,1,27,Bajo exterior sin ascensor,"Berruguete, Madrid"
3,"Ático en Calle de Almansa, 7, Bellas Vistas, M...",1.578€/mes,1,42,Planta 3ª exterior sin ascensor,"Bellas Vistas, Madrid"
4,"Dúplex en Calle Aranjuez, 5, Bellas Vistas, Ma...",1.150€/mes,1,55,Bajo exterior con ascensor,"Bellas Vistas, Madrid"
5,"Piso en Calle Pinos Alta, 42, Ventilla-Almenar...",1.750€/mes,1,65,Planta 1ª exterior con ascensor,"Ventilla-Almenara, Madrid"
6,"Piso en Calle Roble, 26, Cuzco-Castillejos, Ma...",1.400€/mes,1,60,Planta 2ª exterior con ascensor,"Cuzco-Castillejos, Madrid"
7,"Piso en Calle de Almansa, 7, Bellas Vistas, Ma...",1.376€/mes,2,42,Bajo interior sin ascensor,"Bellas Vistas, Madrid"
8,"Ático en Calle de Lorenza Correa, 7, Bellas Vi...",1.100€/mes,1,50,Planta 3ª exterior con ascensor,"Bellas Vistas, Madrid"
9,"Piso en Calle Pinos Alta, Ventilla-Almenara, M...",1.750€/mes,1,50,Planta 1ª exterior con ascensor,"Ventilla-Almenara, Madrid"


In [15]:
df_Pisos.to_csv(r"C:\Users\Ariela\Desktop\EDA - Inmobiliario\EDA_Inmobiliario_Madridv2.csv", index=False, encoding="utf-8-sig")